Import dataset and libraries


In [26]:
import pandas as pd
import tqdm
import numpy as np
import torch
from torch import nn
import os

In [27]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
!find /content/drive/MyDrive -iname "masked_sets*"

/content/drive/MyDrive/masked_sets


In [29]:
from pathlib import Path
data_root = Path("/content/drive/MyDrive/masked_sets")

Prepare data 



In [30]:
from sklearn.model_selection import train_test_split

def load_flat_set(root, class_to_idx):
    # Collect paired RGB and depth paths with labels
    X_rgb, X_depth, y = [], [], []

    rgb_paths = sorted([
        p for p in root.glob("*_rgb_*.png")
        if p.is_file()
    ])

    # For each RGB path, check if the corresponding depth path exists
    for rgb_path in rgb_paths:
        depth_path = Path(str(rgb_path).replace("_rgb_", "_depth_"))

        if depth_path.exists():
            # label is everything before "_random_rgb_"
            label = rgb_path.name.split("_random_rgb_")[0]

            if label in class_to_idx:
                X_rgb.append(str(rgb_path))
                X_depth.append(str(depth_path))
                y.append(class_to_idx[label])
        else:
            print(f"Missing depth for {rgb_path}")

    return X_rgb, X_depth, y

# Path for training set
train_root = data_root / "train"

# Build class list from filenames in the flat train folder
classes = sorted({
    p.name.split("_random_rgb_")[0]
    for p in train_root.glob("*_rgb_*.png")
    if p.is_file()
})
class_to_idx = {c: i for i, c in enumerate(classes)}

# Load training set (paired)
X_rgb_all, X_depth_all, y_all = load_flat_set(train_root, class_to_idx)
print("Loaded training samples:", len(y_all))

# Split training set into train/val
X_rgb_train, X_rgb_val, X_depth_train, X_depth_val, y_train, y_val = train_test_split(
    X_rgb_all, X_depth_all, y_all,
    test_size=0.2,
    random_state=1234,
    stratify=y_all
)
print("Train samples:", len(y_train), "Val samples:", len(y_val))

# Paths for test sets
root_low  = data_root / "test" / "low_occlusion"
root_med  = data_root / "test" / "medium_occlusion"
root_high = data_root / "test" / "high_occlusion"

# Load test sets using the same classes & mapping
X_rgb_test_low,  X_depth_test_low,  y_test_low  = load_flat_set(root_low, class_to_idx)
X_rgb_test_med,  X_depth_test_med,  y_test_med  = load_flat_set(root_med, class_to_idx)
X_rgb_test_high, X_depth_test_high, y_test_high = load_flat_set(root_high, class_to_idx)

# Print test set sizes
print("Low occlusion test samples:", len(y_test_low))
print("Med occlusion test samples:", len(y_test_med))
print("High occlusion test samples:", len(y_test_high))

Loaded training samples: 500
Train samples: 400 Val samples: 100
Low occlusion test samples: 200
Med occlusion test samples: 200
High occlusion test samples: 200


Create RGB and depth CNN model

In [31]:
class SmallCNN(nn.Module):
    def __init__(self, in_channels, num_classes, feat_dim=256):
        super().__init__()
        # Simple CNN backbone
        self.backbone = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1), # 3x3 conv, 32 filters
            nn.ReLU(), 
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), # 3x3 conv, 64 filters
            nn.ReLU(), 
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), # 3x3 conv, 128 filters
            nn.ReLU(), 
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), # 3x3 conv, 256 filters
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)) # global average pooling to get 256-dim features
        )

    def forward(self, x):
        x = self.backbone(x)
        return x.view(x.size(0), -1) # flatten to (batch_size, feat_dim)

Create Fusion Model


In [32]:
import torch
import torch.nn as nn
import torchvision.models as models

class RGBD_Fusion_Model(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # RGB branch
        self.rgb_backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        rgb_feature_dim = self.rgb_backbone.fc.in_features
        self.rgb_backbone.fc = nn.Identity()

        # Depth branch
        self.depth_backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        depth_feature_dim = self.depth_backbone.fc.in_features
        self.depth_backbone.fc = nn.Identity()

        # if depth is 1-channel, change first conv layer
        old_weight = self.depth_backbone.conv1.weight.data
        self.depth_backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.depth_backbone.conv1.weight.data = old_weight.mean(dim=1, keepdim=True)

        # fusion + classifier
        self.classifier = nn.Sequential(
            nn.Linear(rgb_feature_dim + depth_feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, rgb, depth):
        rgb_feat = self.rgb_backbone(rgb)
        depth_feat = self.depth_backbone(depth)

        fused_feat = torch.cat([rgb_feat, depth_feat], dim=1)
        out = self.classifier(fused_feat)
        return out

Train and test CNN models and fuse features from both 

In [33]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for rgb_imgs, depth_imgs, labels in tqdm.tqdm(dataloader):
        rgb_imgs = rgb_imgs.to(device)
        depth_imgs = depth_imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(rgb_imgs, depth_imgs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Total Loss = {total_loss / len(dataloader):.4f}")

def evaluate(model, dataloader, device):
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for rgb_imgs, depth_imgs, labels in dataloader:
            rgb_imgs = rgb_imgs.to(device)
            depth_imgs = depth_imgs.to(device)
            labels = labels.to(device)

            outputs = model(rgb_imgs, depth_imgs)
            predicted = torch.argmax(outputs, dim=1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    acc = 100.0 * correct / total if total > 0 else 0.0
    return acc

In [34]:
# Instantiate separate models for RGB and depth
cnn_rgb = SmallCNN(in_channels=3, num_classes=len(classes)).to(device)
cnn_depth = SmallCNN(in_channels=1, num_classes=len(classes)).to(device)
fusion_model = RGBD_Fusion_Model(num_classes=len(classes)).to(device)

In [35]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import cv2
import numpy as np
import torch

# fixed image size for batching
IMG_SIZE = (224, 224)

# transform for RGB images
rgb_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class RGBDepthDataset(Dataset):
    def __init__(self, rgb_paths, depth_paths, labels, rgb_transform=None):
        self.rgb_paths = rgb_paths
        self.depth_paths = depth_paths
        self.labels = labels
        self.rgb_transform = rgb_transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # load RGB image
        rgb_image = Image.open(self.rgb_paths[idx]).convert("RGB")

        # load depth image as raw grayscale / uint16
        depth_image = cv2.imread(self.depth_paths[idx], cv2.IMREAD_UNCHANGED)
        depth_image = depth_image.astype(np.float32)

        # normalize depth to [0,1] per image
        if depth_image.max() > 0:
            depth_image = depth_image / depth_image.max()

        # resize depth
        depth_image = cv2.resize(depth_image, IMG_SIZE)

        # convert depth to tensor with shape [1,H,W]
        depth_image = torch.tensor(depth_image, dtype=torch.float32).unsqueeze(0)

        # transform RGB
        if self.rgb_transform:
            rgb_image = self.rgb_transform(rgb_image)
        else:
            rgb_image = transforms.ToTensor()(rgb_image)

        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return rgb_image, depth_image, label

# Create Datasets
train_dataset = RGBDepthDataset(X_rgb_train, X_depth_train, y_train, rgb_transform=rgb_transform)
val_dataset = RGBDepthDataset(X_rgb_val, X_depth_val, y_val, rgb_transform=rgb_transform)
test_dataset_low = RGBDepthDataset(X_rgb_test_low, X_depth_test_low, y_test_low, rgb_transform=rgb_transform)
test_dataset_med = RGBDepthDataset(X_rgb_test_med, X_depth_test_med, y_test_med, rgb_transform=rgb_transform)
test_dataset_high = RGBDepthDataset(X_rgb_test_high, X_depth_test_high, y_test_high, rgb_transform=rgb_transform)

# Create DataLoaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader_low = DataLoader(test_dataset_low, batch_size=batch_size, shuffle=False)
test_loader_med = DataLoader(test_dataset_med, batch_size=batch_size, shuffle=False)
test_loader_high = DataLoader(test_dataset_high, batch_size=batch_size, shuffle=False)

# Training loop
train_losses = []
test_accuracies = []
criterion = nn.CrossEntropyLoss()
optimizer_fusion = torch.optim.Adam(fusion_model.parameters(), lr=1e-4)
epochs = 15
for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    train(fusion_model, train_loader, criterion, optimizer_fusion, device)
    acc =evaluate(fusion_model, val_loader, device)
    print(f"Validation Accuracy: {acc:.2f}%")


Epoch 1/15


100%|██████████| 13/13 [00:06<00:00,  1.94it/s]


Total Loss = 3.8628
Validation Accuracy: 10.00%
Epoch 2/15


100%|██████████| 13/13 [00:06<00:00,  2.09it/s]


Total Loss = 3.1916
Validation Accuracy: 34.00%
Epoch 3/15


...

Total Loss = 0.0778
Validation Accuracy: 82.00%
Epoch 14/15


100%|██████████| 13/13 [00:06<00:00,  1.98it/s]


Total Loss = 0.0671
Validation Accuracy: 83.00%
Epoch 15/15


100%|██████████| 13/13 [00:06<00:00,  2.03it/s]


Total Loss = 0.0515
Validation Accuracy: 81.00%


Test the accuracy of the model in low, medium, and high occlusion

In [37]:
acc_low = evaluate(fusion_model, test_loader_low, device)
acc_med = evaluate(fusion_model, test_loader_med, device)
acc_high = evaluate(fusion_model, test_loader_high, device)

print(f"Low Occlusion Accuracy: {acc_low:.2f}%")
print(f"Medium Occlusion Accuracy: {acc_med:.2f}%")
print(f"High Occlusion Accuracy: {acc_high:.2f}%")

Low Occlusion Accuracy: 82.00%
Medium Occlusion Accuracy: 80.00%
High Occlusion Accuracy: 75.00%


Test the model using real scenes